# Aplicación RAG con NVIDIA NIM en Google Colab

Esta versión es **autosuficiente**: configura NVIDIA NIM, carga los documentos, crea un índice Qdrant temporal y publica una interfaz Gradio.

**Modelos:**
- Embeddings: `nvidia/llama-nemotron-embed-vl-1b-v2`
- Generación: `nvidia/nemotron-3.5-lightning-30b-a3b`

> En Colab, ejecuta las celdas en orden. La última celda mostrará una URL pública temporal de Gradio.

---
**Versión modificada para la entrega del workshop** (tareas A1–A4):

- **A2:** corpus propio de *Tostadora Altamira* (10 documentos) en `data/company_documents.json`.
- **A3:** dos prompts del sistema (`PROMPT_ORIGINAL` y `PROMPT_A3`) para comparar antes y después sin reindexar.
- **A4:** el pipeline devuelve también el `vectorstore` y la interfaz muestra el score coseno de cada fuente.


## 1. Instalar dependencias

Después de la instalación no suele ser necesario reiniciar el runtime. Si Colab lo solicita, reinícialo y vuelve a ejecutar desde la celda siguiente.


In [ ]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


## 2. Configurar la API key

En Colab, abre **🔑 Secrets**, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook. También puedes introducirla de forma oculta cuando la celda la solicite.


In [1]:
import os
import json
import getpass
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"
EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2"
COLLECTION = "company_docs_gradio"


from langchain_core.embeddings import Embeddings


class RetryEmbeddings(Embeddings):
    """Reintenta las llamadas de embeddings: el endpoint de NVIDIA a veces responde 502 tras 30 s."""

    def __init__(self, inner, attempts: int = 6, wait: float = 3.0):
        self.inner, self.attempts, self.wait = inner, attempts, wait

    def _retry(self, fn, *args):
        import time
        for attempt in range(1, self.attempts + 1):
            try:
                return fn(*args)
            except Exception as error:
                if attempt == self.attempts:
                    raise
                print(f"  ↻ embeddings: {str(error).splitlines()[0][:30]} · reintento {attempt}")
                time.sleep(self.wait * attempt)

    def embed_documents(self, texts):
        # lotes pequeños: si un lote falla solo se repite ese lote
        out = []
        for i in range(0, len(texts), 32):
            out += self._retry(self.inner.embed_documents, texts[i:i + 32])
        return out

    def embed_query(self, text):
        return self._retry(self.inner.embed_query, text)

print("✅ Configuración NVIDIA NIM lista")


/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Configuración NVIDIA NIM lista


## 3. Cargar los documentos

El corpus propio está en `data/company_documents.json` del repositorio de entrega. La celda lo busca en `data/`, luego en la raíz de la sesión y, si no lo encuentra, lo descarga desde el repo de GitHub (`REPO_RAW_BASE`). Si nada de eso funciona, usa los documentos de demostración.

In [2]:
DEMO_DOCUMENTS = [
    {
        "id": "product_001", "title": "CloudSync Pro — Precios",
        "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.",
        "metadata": {"department": "sales", "type": "pricing"},
    },
    {
        "id": "hr_001", "title": "Política de trabajo remoto",
        "content": "Los empleados pueden trabajar de forma remota hasta 3 días por semana, con aprobación del jefe directo.",
        "metadata": {"department": "hr", "type": "policy"},
    },
    {
        "id": "finance_001", "title": "Política de reembolsos",
        "content": "Los reembolsos mayores a $100 requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original.",
        "metadata": {"department": "finance", "type": "policy"},
    },
    {
        "id": "hr_002", "title": "Onboarding",
        "content": "Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa.",
        "metadata": {"department": "hr", "type": "procedure"},
    },
]


def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


# Cambia esto por tu usuario/repositorio para que Colab descargue el corpus automáticamente.
REPO_RAW_BASE = "https://raw.githubusercontent.com/JeronimoCH/si3003-workshop-rag-JeronimoCH/main"


def find_corpus() -> Path:
    for candidate in (Path("data/company_documents.json"), Path("company_documents.json")):
        if candidate.exists():
            return candidate
    try:
        import requests
        response = requests.get(f"{REPO_RAW_BASE}/data/company_documents.json", timeout=30)
        response.raise_for_status()
        Path("data").mkdir(exist_ok=True)
        Path("data/company_documents.json").write_text(response.text, encoding="utf-8")
        print("⬇ Corpus descargado desde GitHub")
    except Exception as error:
        print(f"ℹ️ No se pudo descargar el corpus ({type(error).__name__})")
    return Path("data/company_documents.json")


company_documents = load_documents(find_corpus())
print(f"✅ {len(company_documents)} documentos disponibles")

for document in company_documents:
    print(f"  - {document['title']} ({document['metadata'].get('department')}, {document['metadata'].get('type')}) · {len(document['content'])} caracteres")

✅ 10 documentos disponibles
  - Catálogo y precios de café 2026 (ventas, pricing) · 586 caracteres
  - Suscripción mensual Altamira Club (ventas, policy) · 545 caracteres
  - Ventas a cafeterías y clientes mayoristas (ventas, procedure) · 586 caracteres
  - Envíos y tiempos de entrega (logistica, policy) · 501 caracteres
  - Política de devoluciones y garantía de frescura (servicio, policy) · 544 caracteres
  - Proceso de tueste y control de calidad (produccion, procedure) · 525 caracteres
  - Compra de café verde a caficultores (produccion, policy) · 500 caracteres
  - Horarios, turnos y trabajo remoto (recursos_humanos, policy) · 444 caracteres
  - Onboarding de nuevos colaboradores (recursos_humanos, procedure) · 474 caracteres
  - Reembolso de gastos y viáticos (finanzas, policy) · 434 caracteres


## 4. Construir el pipeline RAG

Esta celda crea los fragmentos, calcula embeddings con NVIDIA NIM y los indexa en Qdrant. El índice vive en memoria mientras la sesión de Colab esté activa.

Cambios respecto al original:

- **Chunking (A2):** los documentos de Altamira miden entre 430 y 590 caracteres y cada uno trata un solo tema en dos párrafos. Con `chunk_size=400` cada documento se partía en dos chunks (21 en total) y el segundo párrafo quedaba sin el contexto del primero (por ejemplo, «se puede pausar hasta dos meses» sin decir que habla de la suscripción). Con `chunk_size=600` y `chunk_overlap=60` cada documento es un único chunk coherente (10 chunks) y con `k=3` el contexto sigue siendo corto (≈1 500 caracteres).
- **Separación índice / cadena (A3):** `build_index` paga los embeddings una sola vez; `make_pipeline` arma la cadena con cualquier prompt, así se compara el antes y el después sin volver a llamar la API de embeddings.
- **Scores (A4):** se devuelve el `vectorstore` para poder consultar `similarity_search_with_score`.

In [3]:
CHUNK_SIZE = 600
CHUNK_OVERLAP = 60
K = 3
SCORE_THRESHOLD = 0.3

PROMPT_ORIGINAL = (
    "Eres un asistente empresarial. Responde únicamente con información "
    "del contexto proporcionado. Si el contexto no contiene la respuesta, "
    "indica que no tienes esa información. Responde en español y de forma concisa."
)

NO_INFO = "No encuentro esa información en los documentos de Altamira."

PROMPT_A3 = (
    "Eres el asistente interno de Tostadora Altamira. Responde únicamente con "
    "información del CONTEXTO.\n"
    "Reglas obligatorias:\n"
    "1. Cita el título exacto del documento que usaste entre corchetes, por ejemplo "
    "[Política de devoluciones y garantía de frescura].\n"
    f"2. Si el contexto no alcanza para responder, responde exactamente: \"{NO_INFO}\" "
    "y no agregues nada más.\n"
    "3. Si la pregunta es ambigua, da la respuesta para cada caso que cubra el contexto.\n"
    "4. Máximo 3 oraciones, en español."
)

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    # Nemotron 3.5 razona por defecto y ese razonamiento consume max_completion_tokens;
    # sin desactivarlo, muchas respuestas llegan vacías.
    model_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=700,
).with_retry(stop_after_attempt=4)


def build_index(documents: List[Dict], chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP):
    embeddings = RetryEmbeddings(NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END"))

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    # La dimensión se obtiene del modelo y no queda codificada manualmente.
    embedding_dim = len(embeddings.embed_query("prueba de dimensión"))
    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    vectorstore.add_documents(chunks)
    return vectorstore, chunks


def format_docs(items: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
        for item in items
    )


def make_pipeline(vectorstore, system_prompt: str = PROMPT_A3, k: int = K, score_threshold: float = SCORE_THRESHOLD):
    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": k, "score_threshold": score_threshold},
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])
    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return setup_and_retrieval.assign(answer=answer_from_documents)


def build_pipeline(documents: List[Dict], system_prompt: str = PROMPT_A3):
    vectorstore, chunks = build_index(documents)
    return make_pipeline(vectorstore, system_prompt), chunks, vectorstore


print("Construyendo el índice...")
rag_pipeline, indexed_chunks, vectorstore = build_pipeline(company_documents)
titles = [document["title"] for document in company_documents]
print(f"✅ Pipeline listo: {len(titles)} documentos, {len(indexed_chunks)} chunks")

Construyendo el índice...


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


  ↻ embeddings: [502] Bad Gateway · reintento 1


  ↻ embeddings: [502] Bad Gateway · reintento 2


  ↻ embeddings: [502] Bad Gateway · reintento 3


  ↻ embeddings: [502] Bad Gateway · reintento 4


  ↻ embeddings: [502] Bad Gateway · reintento 1


✅ Pipeline listo: 10 documentos, 10 chunks


## 4b. Evaluación por lotes (A2, A3 y A4)

Antes de abrir la interfaz, esta sección corre las preguntas de las tareas y imprime tablas en Markdown listas para pegar en `RESPUESTAS.md`. Cada fila muestra la respuesta y las fuentes con su score coseno (`similarity_search_with_score`, sin umbral, para ver también los scores de las preguntas fuera de contexto).

> La columna «¿Correcta?» la llenas tú comparando con la respuesta esperada.

In [4]:
import time

QUESTIONS_A2 = [
    # (pregunta, respuesta esperada, fuente esperada)
    ("¿Cuánto cuesta la libra de café Reserva?", "$52.000 la libra", "Catálogo y precios de café 2026"),
    ("¿Hasta qué día puedo cancelar la suscripción sin que me cobren el mes siguiente?", "Antes del día 25", "Suscripción mensual Altamira Club"),
    ("¿Qué descuento reciben los clientes mayoristas y cuándo pueden tener crédito?", "18 %; crédito a 30 días tras seis meses de compras puntuales", "Ventas a cafeterías y clientes mayoristas"),
    ("¿Qué pasa con un lote que obtiene menos de 82 puntos en la catación?", "No se vende como Reserva; pasa a la línea Clásica", "Proceso de tueste y control de calidad"),
    ("¿Quién debe aprobar un gasto de viaje de $450.000?", "La gerencia financiera (supera $300.000)", "Reembolso de gastos y viáticos"),
    ("¿Venden máquinas de espresso para el hogar?", "No está en los documentos", "—"),
    ("¿Cuál es el salario de un operario de tostión?", "No está en los documentos", "—"),
]


def sources_with_scores(question: str, k: int = K) -> str:
    hits = vectorstore.similarity_search_with_score(question, k=k)
    return "<br>".join(f"{doc.metadata.get('title')} ({score:.3f})" for doc, score in hits)


def run_table(pipeline, questions, pause: float = 1.0):
    print("| # | Pregunta | Respuesta del RAG | Fuentes (score) | Esperada | ¿Correcta? |")
    print("|---:|---|---|---|---|:---:|")
    for i, (question, expected, _source) in enumerate(questions, 1):
        result = pipeline.invoke(question)
        answer = " ".join(result["answer"].split()).replace("|", "\\|")
        print(f"| {i} | {question} | {answer} | {sources_with_scores(question)} | {expected} | |")
        time.sleep(pause)  # evita errores 429 por ráfagas de llamadas


run_table(rag_pipeline, QUESTIONS_A2)

| # | Pregunta | Respuesta del RAG | Fuentes (score) | Esperada | ¿Correcta? |
|---:|---|---|---|---|:---:|


  ↻ embeddings: [502] Bad Gateway · reintento 1


| 1 | ¿Cuánto cuesta la libra de café Reserva? | $52.000 la libra [Catálogo y precios de café 2026]. | Catálogo y precios de café 2026 (0.401)<br>Suscripción mensual Altamira Club (0.356)<br>Proceso de tueste y control de calidad (0.259) | $52.000 la libra | |


  ↻ embeddings: [502] Bad Gateway · reintento 1


  ↻ embeddings: [502] Bad Gateway · reintento 2


...

  ↻ embeddings: [502] Bad Gateway · reintento 1


| 5 | ¿Quién debe aprobar un gasto de viaje de $450.000? | El gasto de $450.000 requiere aprobación de la gerencia financiera, ya que supera el límite de $300.000 para el líder del área. [Reembolso de gastos y viáticos] | Reembolso de gastos y viáticos (0.336)<br>Suscripción mensual Altamira Club (0.099)<br>Envíos y tiempos de entrega (0.098) | La gerencia financiera (supera $300.000) | |


| 6 | ¿Venden máquinas de espresso para el hogar? | No encuentro esa información en los documentos de Altamira. | Catálogo y precios de café 2026 (0.246)<br>Ventas a cafeterías y clientes mayoristas (0.205)<br>Horarios, turnos y trabajo remoto (0.197) | No está en los documentos | |


  ↻ embeddings: [502] Bad Gateway · reintento 1


| 7 | ¿Cuál es el salario de un operario de tostión? | No encuentro esa información en los documentos de Altamira. | Horarios, turnos y trabajo remoto (0.318)<br>Proceso de tueste y control de calidad (0.245)<br>Reembolso de gastos y viáticos (0.223) | No está en los documentos | |


### A3 · Prompt antes y después

Se reutiliza el mismo índice: solo cambia el prompt del sistema. Las tres preguntas son una fácil, una ambigua y una fuera de contexto.

In [5]:
QUESTIONS_A3 = [
    ("¿Cuánto cuesta la libra de café Reserva?", "$52.000 la libra", "fácil"),
    ("¿Cuánto cuesta el envío?", "$9.000 en el Valle de Aburrá (gratis > $120.000); $16.000 resto del país", "ambigua"),
    ("¿Venden máquinas de espresso para el hogar?", "No está en los documentos", "fuera de contexto"),
]

pipeline_before = make_pipeline(vectorstore, PROMPT_ORIGINAL)
pipeline_after = make_pipeline(vectorstore, PROMPT_A3)

print("| Tipo | Pregunta | Antes (PROMPT_ORIGINAL) | Después (PROMPT_A3) |")
print("|---|---|---|---|")
for question, _expected, kind in QUESTIONS_A3:
    before = " ".join(pipeline_before.invoke(question)["answer"].split()).replace("|", "\\|")
    time.sleep(1)
    after = " ".join(pipeline_after.invoke(question)["answer"].split()).replace("|", "\\|")
    time.sleep(1)
    print(f"| {kind} | {question} | {before} | {after} |")

| Tipo | Pregunta | Antes (PROMPT_ORIGINAL) | Después (PROMPT_A3) |
|---|---|---|---|


  ↻ embeddings: [502] Bad Gateway · reintento 1


  ↻ embeddings: [502] Bad Gateway · reintento 2


  ↻ embeddings: [502] Bad Gateway · reintento 3


  ↻ embeddings: [502] Bad Gateway · reintento 1


...

  ↻ embeddings: [502] Bad Gateway · reintento 1


  ↻ embeddings: [502] Bad Gateway · reintento 2


| ambigua | ¿Cuánto cuesta el envío? | El costo del envío varía según la ubicación y el monto de la compra: * **Dentro del Valle de Aburrá:** Cuesta $9.000, pero es **gratis** en compras mayores a $120.000. * **Resto de Colombia:** Cuesta $16.000 y tiene un tiempo de entrega de 2 a 4 días hábiles. No se realizan envíos internacionales. | El costo del envío varía según la ubicación y el monto de la compra. Dentro del Valle de Aburrá, la entrega cuesta $9.000, pero es gratis en compras mayores a $120.000. Para el resto de Colombia, el envío tiene un costo fijo de $16.000. |


  ↻ embeddings: [502] Bad Gateway · reintento 1


| fuera de contexto | ¿Venden máquinas de espresso para el hogar? | No, soy un modelo de lenguaje y no tengo información sobre la venta de máquinas de espresso para el hogar en el contexto proporcionado. | No encuentro esa información en los documentos de Altamira. |


### A4 · Scores para elegir `score_threshold`

Tres preguntas que sí se responden y dos fuera de contexto. Se muestran los 3 mejores scores de cada una.

In [6]:
QUESTIONS_A4 = [
    ("¿Cuánto cuesta la libra de café Reserva?", "buena"),
    ("¿Qué incluye la primera semana de un colaborador nuevo?", "buena"),
    ("¿Cuántos días tarda un envío a Bogotá?", "buena"),
    ("¿Venden máquinas de espresso para el hogar?", "fuera"),
    ("¿Quién ganó el mundial de fútbol de 2022?", "fuera"),
]

print("| Pregunta | Tipo | top-1 | top-2 | top-3 |")
print("|---|---|---|---|---|")
for question, kind in QUESTIONS_A4:
    hits = vectorstore.similarity_search_with_score(question, k=3)
    cols = " | ".join(f"{score:.3f} · {doc.metadata['title']}" for doc, score in hits)
    print(f"| {question} | {kind} | {cols} |")

| Pregunta | Tipo | top-1 | top-2 | top-3 |
|---|---|---|---|---|


| ¿Cuánto cuesta la libra de café Reserva? | buena | 0.401 · Catálogo y precios de café 2026 | 0.356 · Suscripción mensual Altamira Club | 0.259 · Proceso de tueste y control de calidad |


  ↻ embeddings: [502] Bad Gateway · reintento 1


| ¿Qué incluye la primera semana de un colaborador nuevo? | buena | 0.529 · Onboarding de nuevos colaboradores | 0.260 · Horarios, turnos y trabajo remoto | 0.132 · Suscripción mensual Altamira Club |


  ↻ embeddings: [502] Bad Gateway · reintento 1


| ¿Cuántos días tarda un envío a Bogotá? | buena | 0.490 · Envíos y tiempos de entrega | 0.364 · Política de devoluciones y garantía de frescura | 0.349 · Ventas a cafeterías y clientes mayoristas |


  ↻ embeddings: [502] Bad Gateway · reintento 1


| ¿Venden máquinas de espresso para el hogar? | fuera | 0.248 · Catálogo y precios de café 2026 | 0.203 · Ventas a cafeterías y clientes mayoristas | 0.199 · Suscripción mensual Altamira Club |


| ¿Quién ganó el mundial de fútbol de 2022? | fuera | 0.029 · Compra de café verde a caficultores | 0.024 · Proceso de tueste y control de calidad | 0.023 · Envíos y tiempos de entrega |


## 5. Crear la interfaz Gradio

In [7]:
def chat(question: str, history: list | None):
    history = history or []
    question = (question or "").strip()
    if not question:
        return "", history, "Escribe una pregunta."

    try:
        result = rag_pipeline.invoke(question)
        answer = result["answer"].strip()

        # A4: los scores se consultan sin umbral para ver también los descartados.
        scored = vectorstore.similarity_search_with_score(question, k=K)
        lines = []
        for document, score in scored:
            used = "✅" if score >= SCORE_THRESHOLD else "⛔ bajo el umbral"
            lines.append(
                f"- {document.metadata.get('title', 'Sin título')} "
                f"(chunk {document.metadata.get('chunk_index')}) — score **{score:.3f}** {used}"
            )
        sources_md = (
            f"**Fuentes recuperadas (coseno, umbral {SCORE_THRESHOLD}):**\n" + "\n".join(lines)
            if lines else "No se encontraron documentos relevantes."
        )

    except Exception as error:
        answer = "Ocurrió un error al consultar el pipeline. Revisa el detalle mostrado debajo."
        sources_md = f"**Error:** `{type(error).__name__}: {error}`"

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return "", history, sources_md


def clear_chat():
    return [], ""


with gr.Blocks(title="RAG Tostadora Altamira") as demo:
    gr.Markdown(
        "# RAG Tostadora Altamira\n"
        "Consulta los documentos internos de la tostadora mediante **LangChain + NVIDIA NIM + Qdrant**."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Documentos indexados")
            gr.Markdown("\n".join(f"- {title}" for title in titles))
            gr.Markdown(f"*{len(indexed_chunks)} chunks en el índice*")
            gr.Markdown("---")
            gr.Markdown(
                f"**Embeddings:** `{EMBEDDING_MODEL}`  \n"
                f"**LLM:** `{GENERATION_MODEL}`  \n"
                "**Vector store:** Qdrant en memoria  \n"
                f"**Retrieval:** k={K}, threshold={SCORE_THRESHOLD}, chunk_size={CHUNK_SIZE}"
            )

        with gr.Column(scale=2):
            # Gradio 6 usa mensajes {role, content} por defecto;
            # el antiguo argumento type="messages" ya no existe.
            chatbot = gr.Chatbot(height=430, label="Chat")
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Haz una pregunta sobre los documentos...",
                    label="Pregunta",
                    lines=2,
                    scale=4,
                )
                send_button = gr.Button("Enviar", variant="primary", scale=1)

            sources_box = gr.Markdown()
            clear_button = gr.Button("Limpiar", variant="secondary")

    gr.Examples(
        examples=[
            ["¿Cuánto cuesta la libra de café Reserva?"],
            ["¿Hasta qué día puedo cancelar la suscripción sin que me cobren el mes siguiente?"],
            ["¿Qué pasa con un lote que obtiene menos de 82 puntos en la catación?"],
            ["¿Cuánto cuesta el envío?"],
            ["¿Venden máquinas de espresso para el hogar?"],
        ],
        inputs=question_box,
        label="Preguntas de ejemplo",
    )

    send_button.click(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    question_box.submit(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    clear_button.click(clear_chat, outputs=[chatbot, sources_box])


## 6. Ejecutar la aplicación en Colab

`share=True` crea el enlace público temporal. La aplicación estará disponible mientras esta celda y el runtime de Colab continúen activos.


In [ ]:
demo.queue().launch(
    share=True,
    debug=True,
    show_error=True,
)


### Detener o volver a ejecutar

- Interrumpe la última celda para detener el servidor.
- Antes de volver a ejecutarla, puedes cerrar instancias previas con `gr.close_all()`.
- El enlace público cambia cada vez que se inicia la aplicación.


In [ ]:
# Ejecuta esta celda solo cuando quieras cerrar servidores Gradio anteriores.
gr.close_all()
